In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

In [0]:
import sys
sys.path.append("../src")
from pyspark.sql import functions as F
from observability.metrics import RowCountMetric, NullRateMetric, DistinctCountMetric
from observability.runner import ObservabilityRunner
import time
import statistics
import datetime
import uuid
from observability.config import METRIC_HISTORY_TABLE, ANOMALY_HISTORY_TABLE, PERFORMANCE_HISTORY_TABLE
from observability.anomalies import RowCountAnomalyDetector, NullRateAnomalyDetector, PerformanceRunTimeAnomalyDetector
from observability.anomalies_runner import AnomalyDetectorRunner


In [0]:

metrics_config = [
    {
        "metric": "row_count",
        "enabled": True
    },
    {
        "metric": "null_rate",
        "enabled": True,
        "column_name": "amount"
    },
    {
        "metric": "distinct_count",
        "enabled": True,
        "column_name": "country"
    }
]

metric_classes = {
    "row_count": RowCountMetric,
    "null_rate": NullRateMetric,
    "distinct_count": DistinctCountMetric
}

metrics = []

for item in metrics_config:

    if not item["enabled"]:
        continue
    elif item["enabled"] and "column_name" not in item:
        metric_name = metric_classes[item["metric"]]()
        metrics.append(metric_name)
    elif item["enabled"] and "column_name" in item:
        metric_name = metric_classes[item["metric"]](item["column_name"])
        metrics.append(metric_name)

ANOMALY_THRESHOLD = {
    "row_count": 30, # (percent)
    "null_rate": 10, # max null rate (percent)
    "runtime_performance": 50, # (percent)
}

anomalies = [
    RowCountAnomalyDetector,
    NullRateAnomalyDetector
]

anomalies_type = {
    RowCountAnomalyDetector: "row_count",
    NullRateAnomalyDetector: "null_rate",
    PerformanceRunTimeAnomalyDetector: "runtime_performance",
}
        

In [0]:
orders_df = (
            spark.range(1_050_000)
            .withColumnRenamed("id", "order_id")
            .withColumn(
                "country",
                F.when(F.col("order_id") % 10 == 0, None)
                .when(F.col("order_id") % 3 == 0, "PL")
                .when(F.col("order_id") % 3 == 1, "DE")
                .otherwise("UK")
                )
            .withColumn(
                "amount",
                F.when(F.col("order_id") % 3 == 0, None)
                .otherwise((F.rand() * 1000).cast("double"))
            )
            )

display(orders_df)
orders_df.printSchema()
orders_df.select(F.spark_partition_id().alias("partition_id")).groupBy("partition_id").count().orderBy("partition_id").show()

In [0]:
table_name = "orders"

observability_runner_v2 = ObservabilityRunner(metrics, orders_df)

start=time.perf_counter()

run_result = observability_runner_v2.run().collect()

end=time.perf_counter()

run_result = run_result[0].asDict()

run_id_metric = str(uuid.uuid4())

timestamp = datetime.datetime.now()

result_time = end-start

result = {
    "run_id": run_id_metric,
    "table_name": table_name,
    **run_result,
    "executed_at": timestamp
}

ddl_schema = "run_id STRING NOT NULL, table_name STRING, row_count LONG, null_rate_amount DOUBLE, distinct_count_country INT, executed_at TIMESTAMP"

result_df = (
    spark.createDataFrame([result], 
                          schema=ddl_schema)
    )

display(result_df)

result_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(METRIC_HISTORY_TABLE)

In [0]:
table_name = "orders"

anomaly_run = AnomalyDetectorRunner(
    spark = spark, 
    METRIC_HISTORY_TABLE = METRIC_HISTORY_TABLE, 
    PERFORMANCE_HISTORY_TABLE = PERFORMANCE_HISTORY_TABLE, 
    ANOMALIES = anomalies, 
    PERFORMANCE_ANOMALY = PerformanceRunTimeAnomalyDetector, 
    thresholds = ANOMALY_THRESHOLD
    )

anomaly_result = anomaly_run.run_anomaly_detector()

display(anomaly_result)

anomaly_results = [
    row.asDict()
    for row in spark.createDataFrame(anomaly_result).collect()
    ]

run_id = run_id_metric

timestamp = datetime.datetime.now()

anomaly_history_result = []

for anomaly in anomaly_results:
    anomaly_history_result.append({
        "run_id": run_id,
        "table_name": table_name,
        **anomaly,
        "executed_at": timestamp
        })

ddl_schema_anomaly = """
                    run_id STRING NOT NULL,
                    table_name STRING,
                    anomaly_type STRING NOT NULL,
                    is_anomaly BOOLEAN,
                    current_value DOUBLE,
                    previous_value DOUBLE,
                    percentage_change DOUBLE,
                    threshold DOUBLE,
                    executed_at TIMESTAMP
                    """

anomaly_history_result_df = spark.createDataFrame(anomaly_history_result, ddl_schema_anomaly)

anomaly_history_result_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(ANOMALY_HISTORY_TABLE)

In [0]:
performance_result = {
        "run_id": run_id_metric,
        "table_name": table_name,
        "runtime_seconds": result_time,
        "row_count": run_result["row_count"],
        "executed_at": timestamp
}

ddl_schema_performance_result = """
                                run_id STRING NOT NULL,
                                table_name STRING,
                                runtime_seconds DOUBLE,
                                row_count LONG,
                                executed_at TIMESTAMP
                                """

performance_result_df = spark.createDataFrame([performance_result], ddl_schema_performance_result)

performance_result_df.write \
                    .format("delta") \
                    .mode("append") \
                    .saveAsTable(PERFORMANCE_HISTORY_TABLE)